Calculate distances between all pairs of species

In [ ]:
from Bio import Phylo
import numpy as np

# from scipy.spatial.distance import pdist, squareform

# Load phylogenetic tree
tree = Phylo.read(
    r"..\..\input\aligned_caffeine_without_nodes_w_no_coords_tree.nwk", "newick"
)


# Extract terminal node names (species)
terminals = tree.get_terminals()
n = len(terminals)

# Initialize an empty distance matrix
phylo_distances = np.zeros((n, n))

# Compute pairwise distances
for i, sp1 in enumerate(terminals):
    for j, sp2 in enumerate(terminals):
        phylo_distances[i, j] = tree.distance(sp1, sp2)

# Optionally, print or convert the matrix to a DataFrame for better readability
import pandas as pd

species_names = [str(sp) for sp in terminals]
phylo_df = pd.DataFrame(phylo_distances, index=species_names, columns=species_names)
print(phylo_df)

                         C_kianjavatensis_A602  C_bertrandii_A5  \
C_kianjavatensis_A602                  0.00000          0.13751   
C_bertrandii_A5                        0.13751          0.00000   
C_richardii_A575                       0.12301          0.11224   
C_farafanganensis_A208                 0.13209          0.12132   
C_millotii_A222                        0.12205          0.11128   
C_abbayesii_A601                       0.11616          0.10539   
C_homollei_A945                        0.11978          0.11299   
C_perrieri_A12                         0.13163          0.12484   
C_leroyi_A315                          0.11986          0.11307   
C_andrambovatensis_A310                0.13703          0.13024   
C_vianneyi_A946                        0.12842          0.12301   
C_resinosa_A8                          0.13620          0.13079   
C_arenesiana_A403                      0.14402          0.13861   
C_vatovavyensis_A830                   0.14993          0.1445

Calculate geographic distances between species locations

In [ ]:
import pandas as pd
import numpy as np
from geopy.distance import geodesic  # For Haversine-based distances

# Load CSV file
geo_df = pd.read_csv(r"..\data\dr_guyot_all_for_collection.csv")
print(geo_df.head())

        specimen_id   latitude  longitude
0  Coffea_abbayesii -24.366660  46.450000
1  Coffea_abbayesii -24.730000  46.830000
2  Coffea_abbayesii -24.733300  46.833300
3  Coffea_abbayesii -24.666670  46.833330
4  Coffea_abbayesii -24.733333  46.833333


Only one  geolocation point per species is necessary to calculate mantel test. Exercise to be done with median, mean, hand-picked


In [ ]:
# Group by species and calculate median coordinates
median_coords = (
    geo_df.groupby("specimen_id")[["longitude", "latitude"]].median().reset_index()
)
print(median_coords)

                 specimen_id  longitude   latitude
0           Coffea_abbayesii  46.962222 -24.733333
1         Coffea_ambanjensis  48.317222 -13.829722
2         Coffea_ambongensis  46.418331 -15.576389
3    Coffea_andrambovatensis  47.758333 -21.516667
4        Coffea_ankaranensis  49.200000 -12.883333
..                       ...        ...        ...
118    Coffea_vavateninensis  49.016667 -17.433333
119          Coffea_vianneyi  47.866667 -21.382389
120      Coffea_vohemarensis  49.571945 -13.593056
121         Coffea_wightiana  79.027038  11.734040
122      Coffea_zanguebariae  38.516660  -8.683330

[123 rows x 3 columns]


In [ ]:
# Group by species and calculate mean coordinates
mean_coords = (
    geo_df.groupby("specimen_id")[["longitude", "latitude"]].mean().reset_index()
)
print(mean_coords)

                 specimen_id  longitude   latitude
0           Coffea_abbayesii  47.108862 -24.072093
1         Coffea_ambanjensis  48.309490 -13.896019
2         Coffea_ambongensis  46.330912 -15.631032
3    Coffea_andrambovatensis  47.791667 -21.475000
4        Coffea_ankaranensis  49.090711 -13.539108
..                       ...        ...        ...
118    Coffea_vavateninensis  49.016667 -17.433333
119          Coffea_vianneyi  47.866167 -21.381204
120      Coffea_vohemarensis  49.111216 -16.180804
121         Coffea_wightiana  78.559811  11.471653
122      Coffea_zanguebariae  36.847569 -12.462594

[123 rows x 3 columns]


KeyError: "['C_dolichophylla_A206', 'C_ambodirianensis_A572', 'C_andrambovatensis_A310', 'C_vatovavyensis_A830', 'C_mauritiana_Makes4', 'C_mauritiana_BM17_25'] not in index"

In [ ]:
from sklearn.cluster import DBSCAN

coords_df = pd.read_csv("../data/dr_guyot_all_for_collection.csv")
# coords_df['specimen_id'] = coords_df['specimen_id'].str.replace(' ', '_')

# Only keep necessary columns
coords_df = coords_df[["specimen_id", "latitude", "longitude"]].dropna()


# Function to compute centroid of largest cluster per species
def compute_cluster_centroids(df):
    centroids = []
    for name, group in df.groupby("specimen_id"):
        coords = group[["latitude", "longitude"]].values

        if len(coords) < 3:
            # If fewer than 3 points, skip clustering and just take mean
            centroid = coords.mean(axis=0)
        else:
            # Apply DBSCAN to find clusters
            db = DBSCAN(eps=0.5, min_samples=2).fit(coords)
            labels = db.labels_
            if np.all(labels == -1):
                # All outliers, fall back to mean
                centroid = coords.mean(axis=0)
            else:
                # Find largest cluster
                largest_label = pd.Series(labels).value_counts().idxmax()
                cluster_points = coords[labels == largest_label]
                centroid = cluster_points.mean(axis=0)

        centroids.append(
            {"specimen_id": name, "latitude": centroid[0], "longitude": centroid[1]}
        )

    return pd.DataFrame(centroids)


# Compute cluster-based centroids
centroid_coords_df = compute_cluster_centroids(coords_df)

In [73]:
centroid_coords_df

,specimen_id,latitude,longitude
0,Coffea_abbayesii,-24.744807,46.919542
1,Coffea_ambanjensis,-13.896019,48.309490
2,Coffea_ambongensis,-15.576512,46.417994
3,Coffea_andrambovatensis,-21.475000,47.791667
4,Coffea_ankaranensis,-12.754686,49.213115
...,...,...,...
118,Coffea_vavateninensis,-17.433333,49.016667
119,Coffea_vianneyi,-21.381204,47.866167
120,Coffea_vohemarensis,-13.582292,49.732917
121,Coffea_wightiana,11.173449,78.367712


In [ ]:
import pandas as pd

# Load coordinate metadata
coords_df = pd.read_csv("../data/dr_guyot_all_for_collection.csv")
genetic_df = pd.read_csv("../data/genetic_dist_matrix.csv", index_col=0)

# Extract tree labels
tree_labels = genetic_df.index.tolist()

# Create mapping from simplified label to tree label
# E.g., match Coffea_abbayesii to C_abbayesii_A601_1, etc.
parsed_data = []
for label in tree_labels:
    parts = label.split("_")
    accession = parts[-2] if parts[-1].isdigit() else parts[-1]
    species_core = parts[1]
    species_full = f"Coffea_{species_core}"
    parsed_data.append({"tree_label": label, "match_label": species_full})

label_map = pd.DataFrame(parsed_data)

# Add a unique index to each occurrence of the same specimen
coords_df["count"] = coords_df.groupby("specimen_id").cumcount() + 1

# Generate unique labels that should match tree labels (C_species_accession_index)
coords_df["species_core"] = coords_df["specimen_id"].str.extract(r"Coffea_(\w+)")
coords_df["accession"] = coords_df["specimen_id"].str.extract(r"\(([^)]+)\)")
coords_df["tree_label"] = (
    "C_"
    + coords_df["species_core"]
    + "_"
    + coords_df["accession"]
    + "_"
    + coords_df["count"].astype(str)
)

# Filter only rows that are in the genetic distance matrix
filtered_coords = coords_df[coords_df["tree_label"].isin(tree_labels)]

# Save the result
output_path = "/mnt/data/dr_guyot_collection_specim_renamed.csv"
# filtered_coords.to_csv(output_path, index=False)

filtered_coords[["tree_label", "latitude", "longitude"]].head()

,tree_label,latitude,longitude


In [ ]:
genetic_df_clean = df.reset_index()
genetic_df_clean.columns = ["tree_label"] + list(genetic_df_clean.columns[1:])
genetic_df_clean["tree_label"] = genetic_df_clean["tree_label"].str.replace(" ", "_")
genetic_df_clean["species_core"] = genetic_df_clean["tree_label"].str.split("_").str[1]
genetic_df_clean["match_label"] = "Coffea_" + genetic_df_clean["species_core"]

# Step 2: Merge with geo_df
geo_df_full = centroid_coords_df.copy()  # tried with median coords with poor results
geo_df_full = geo_df_full.merge(
    genetic_df_clean[["tree_label", "match_label"]],
    left_on="specimen_id",
    right_on="match_label",
    how="left",
)

# Step 3: Drop old IDs, keep only clean tree-labeled data
geo_df_final = geo_df_full.drop(columns=["specimen_id", "match_label"])
geo_df_final = geo_df_final.dropna(subset=["tree_label"])  # Keep only matched rows
geo_df_final = geo_df_final[["tree_label", "latitude", "longitude"]]
geo_df_final_renamed = geo_df_final.rename(columns={"tree_label": "specimen_id"})

geo_df_final_renamed.to_csv("../data/dr_guyot_all_for_collection_renamed.csv")
geo_df_final_renamed

,specimen_id,latitude,longitude
0,C_abbayesii_A601,-24.744807,46.919542
3,C_andrambovatensis_A310,-21.475000,47.791667
6,C_arenesiana_A403,-19.023884,48.067106
10,C_bertrandii_A5,-25.037400,46.653250
31,C_dubardii_A969,-12.759268,49.141778
35,C_farafanganensis_A208,-21.382741,47.864843
40,C_heimii_A516,-12.418727,49.381558
42,C_homollei_A945,-21.382917,47.865278
44,C_humbertii_RNF785,-23.420697,44.101995
50,C_kianjavatensis_A602,-21.383820,47.878680


In [ ]:
from geopy.distance import geodesic

# Ensure latitude and longitude are numeric
geo_coords = geo_df_final_renamed[["specimen_id", "latitude", "longitude"]].copy()
geo_coords["latitude"] = pd.to_numeric(geo_coords["latitude"], errors="coerce")
geo_coords["longitude"] = pd.to_numeric(geo_coords["longitude"], errors="coerce")

# Drop rows with missing coordinates
geo_coords.dropna(subset=["latitude", "longitude"], inplace=True)

# Set specimen_id as index
geo_coords.set_index("specimen_id", inplace=True)

# Initialize empty distance matrix
specimens = geo_coords.index.tolist()
geo_distances = pd.DataFrame(index=specimens, columns=specimens, dtype=float)

# Compute pairwise geographic distances using geodesic
for i in range(len(specimens)):
    for j in range(i, len(specimens)):  # Only upper triangle
        id_i = specimens[i]
        id_j = specimens[j]
        coord1 = (geo_coords.loc[id_i, "latitude"], geo_coords.loc[id_i, "longitude"])
        coord2 = (geo_coords.loc[id_j, "latitude"], geo_coords.loc[id_j, "longitude"])
        distance = geodesic(coord1, coord2).kilometers
        geo_distances.loc[id_i, id_j] = distance
        geo_distances.loc[id_j, id_i] = distance  # Mirror value

In [77]:
print(geo_distances.head())
geo_distances.to_csv("../data/geographic_distances_full.csv")

                         C_abbayesii_A601  C_andrambovatensis_A310  \
C_abbayesii_A601                 0.000000               372.970662   
C_andrambovatensis_A310        372.970662                 0.000000   
C_arenesiana_A403              644.466606               272.878470   
C_bertrandii_A5                 42.123524               411.358702   
C_dubardii_A969               1347.142265               975.205876   

                         C_arenesiana_A403  C_bertrandii_A5  C_dubardii_A969  \
C_abbayesii_A601                644.466606        42.123524      1347.142265   
C_andrambovatensis_A310         272.878470       411.358702       975.205876   
C_arenesiana_A403                 0.000000       681.674431       702.709879   
C_bertrandii_A5                 681.674431         0.000000      1384.046663   
C_dubardii_A969                 702.709879      1384.046663         0.000000   

                         C_farafanganensis_A208  C_heimii_A516  \
C_abbayesii_A601                

Performing Mantel test

In [ ]:
from skbio.stats.distance import mantel, DistanceMatrix
import pandas as pd

# Load data
genetic_df = pd.read_csv("../data/genetic_dist_matrix.csv", index_col=0)
geo_df = pd.read_csv("../data/geographic_distances_full.csv", index_col=0)
metadata_df = pd.read_csv("../data/data_w_series_habitat.csv")

In [79]:
print(genetic_df)

                         C_kianjavatensis_A602  C_bertrandii_A5  \
C_kianjavatensis_A602                      0.0              2.0   
C_bertrandii_A5                            2.0              0.0   
C_richardii_A575                           2.0              4.0   
C_farafanganensis_A208                     2.0              4.0   
C_millotii_A222                            2.0              4.0   
C_abbayesii_A601                           2.0              4.0   
C_homollei_A945                            2.0              4.0   
C_perrieri_A12                             2.0              4.0   
C_leroyi_A315                              2.0              4.0   
C_andrambovatensis_A310                    2.0              4.0   
C_vianneyi_A946                            2.0              4.0   
C_resinosa_A8                              2.0              4.0   
C_arenesiana_A403                          2.0              4.0   
C_vatovavyensis_A830                       2.0              4.

In [ ]:
shared_species = sorted(
    set(genetic_df.index) & set(geo_df.index) & set(metadata_df["Species_name"])
)
genetic_df = genetic_df.loc[shared_species, shared_species]
geo_df = geo_df.loc[shared_species, shared_species]
metadata_df = metadata_df[metadata_df["Species_name"].isin(shared_species)]
metadata_df.to_csv("../data/matched_specimen_metadata.csv", index=False)

In [81]:
metadata_df

,Species_name,caffeine_percent,Population code,Species,Botanical series,Origin/locality,Province,Latitude,Longitude,Habitat class
0,C_tetragona_A252,0.030,A252,C. tetragona Jum. & H.Perrier,Garcinioides,Behangony (Est de Maromandia),Mahajanga (Northwest),14°13′30″ S,48°09′ E,3.0
1,C_dubardii_A969,0.000,A969,C. dubardii Jum.,Garcinioides,North of Vohemar (Diégo-Suarez),Antsiranana (North),13°20′ S,49°57′ E,3.0
2,C_heimii_A516,0.000,A516,C. heimii J.-F.Leroy,Garcinioides,Diégo-Suarez (Sahafary),Antsiranana (North),12°35′ S,49°26′30″ E,3.0
3,C_farafanganensis_A208,0.045,A208,C. farafanganensis J.-F.Leroy,Millotii complex,Farafangana (Amboangibe),Fianarantsoa (Southeast),22°53′ S,47°48′ E,1.0
6,C_richardii_A575,0.030,A575,C. richardii J.-F.Leroy,Millotii complex,Fenerive-Est (Tampolo),Toamasina (East),17°17′ S,49°25′ E,1.0
7,C_abbayesii_A601,0.000,A601,C. abbayesii J.-F.Leroy,Millotii complex,Fort-Dauphin (Isaka-Ivondro),Toliara (Southeast),24°45′15″ S,46°51′45″ E,2.0
8,C_millotii_A222,0.000,A222,C. millotii J.-F.Leroy,Millotii complex,Mananjary (Tolongoina),Fianarantsoa (Southeast),21°31′ S,47°25′ E,2.0
9,C_leroyi_A315,0.020,A315,C. leroyi A.P.Davis,Multiflorae,Ifanadiana (Ambodiarafia),Fianarantsoa (Southeast),21°20′ S,47°45′ E,1.0
10,C_andrambovatensis_A310,0.000,A310,C. andrambovatensis J.-F.Leroy,Multiflorae,Vatovavy,Fianarantsoa (Southeast),21°24′3″ S,47°56′32″ E,1.0
11,C_resinosa_A8,0.000,A8,C. resinosa (Hook.f.) Radlk.,Multiflorae,Nosy Varika,Fianarantsoa (Southeast),20°36′26.6″ S,48°31′56.5″ E,1.0


In [ ]:
print("\nSpecies count per botanical series (after alignment):")
for series, group in metadata_df.groupby("Botanical series"):
    print(f"{series}: {len(group)} species")

# Run Mantel tests by series with at least 3 species
results = []
for series, group in metadata_df.groupby("Botanical series"):
    species = group["Species_name"].tolist()
    if len(species) < 3:
        continue

    g_dm = DistanceMatrix(genetic_df.loc[species, species].values, ids=species)
    d_dm = DistanceMatrix(geo_df.loc[species, species].values, ids=species)

    r, p, _ = mantel(g_dm, d_dm, method="pearson", permutations=10000)
    results.append(
        {
            "Botanical series": series,
            "n_species": len(species),
            "Mantel_r": round(r, 3),
            "p_value": round(p, 4),
        }
    )

# Output results
mantel_df = pd.DataFrame(results)

if "p_value" in mantel_df.columns:
    print("\nMantel Test Results:")
    print(mantel_df.sort_values("p_value"))
else:
    print("\nNo valid series with ≥3 species found for Mantel testing.")
# Output results
mantel_df = pd.DataFrame(results)
print(mantel_df.sort_values("p_value"))


Species count per botanical series (after alignment):
Garcinioides: 3 species
Millotii complex: 4 species
Multiflorae: 7 species
Subterminales: 3 species
Verae: 3 species

Mantel Test Results:
   Botanical series  n_species  Mantel_r  p_value
1  Millotii complex          4     0.635   0.1667
2       Multiflorae          7     0.335   0.2838
0      Garcinioides          3     0.999   0.3302
4             Verae          3     0.501   0.6632
3     Subterminales          3     0.104   1.0000
   Botanical series  n_species  Mantel_r  p_value
1  Millotii complex          4     0.635   0.1667
2       Multiflorae          7     0.335   0.2838
0      Garcinioides          3     0.999   0.3302
4             Verae          3     0.501   0.6632
3     Subterminales          3     0.104   1.0000


mantel for each sample

In [ ]:
# Re-load the genetic distance matrix
genetic_df = pd.read_csv("../data/genetic_dist_matrix.csv", index_col=0)

# Rebuild tree label mapping
tree_labels = genetic_df.index.tolist()
tree_mapping = {}
for label in tree_labels:
    parts = label.split("_")
    if len(parts) >= 3:
        species_name = parts[1].lower()
        tree_mapping[species_name] = label

# Re-map the coordinate file with unique specimen names
coords_df = pd.read_csv("../data/dr_guyot_all_for_collection.csv")
coords_df_clean = coords_df.dropna(subset=["latitude", "longitude"]).copy()
coords_df_clean["species_key"] = (
    coords_df_clean["specimen_id"].str.replace("Coffea_", "", regex=False).str.lower()
)
coords_df_clean["tree_label"] = coords_df_clean["species_key"].map(tree_mapping)
matched_coords = coords_df_clean.dropna(subset=["tree_label"])

# Generate unique specimen labels
counter = {}
unique_ids = []
for label in matched_coords["tree_label"]:
    if label not in counter:
        counter[label] = 1
    else:
        counter[label] += 1
    unique_label = f"{label}_{counter[label]}"
    unique_ids.append(unique_label)

matched_coords["unique_specimen_id"] = unique_ids
coords_for_matrix = matched_coords[
    ["unique_specimen_id", "latitude", "longitude"]
].set_index("unique_specimen_id")

# Compute optimized haversine distance matrix
coords_rad = np.radians(coords_for_matrix[["latitude", "longitude"]].values)
R = 6371.0

lat1 = coords_rad[:, 0][:, np.newaxis]
lat2 = coords_rad[:, 0]
delta_lat = lat1 - lat2

lon1 = coords_rad[:, 1][:, np.newaxis]
lon2 = coords_rad[:, 1]
delta_lon = lon1 - lon2

a = (
    np.sin(delta_lat / 2.0) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2.0) ** 2
)
c = 2 * np.arcsin(np.sqrt(a))
distances = R * c

# Create DataFrame and save
specimen_ids = coords_for_matrix.index
distance_matrix_optimized = pd.DataFrame(
    distances, index=specimen_ids, columns=specimen_ids
)
distance_matrix_optimized.to_csv(
    "../data/geographic_dist_matrix_full_specimens_optimized.csv"
)

distance_matrix_optimized.iloc[:5, :5]

C:\Users\adm1\AppData\Local\Temp\ipykernel_39384\4099715395.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  matched_coords['unique_specimen_id'] = unique_ids


unique_specimen_id,C_abbayesii_A601_1,C_abbayesii_A601_2,C_abbayesii_A601_3,C_abbayesii_A601_4,C_abbayesii_A601_5
unique_specimen_id,,,,,
C_abbayesii_A601_1,0.000000,55.762975,56.258536,51.155157,56.263492
C_abbayesii_A601_2,55.762975,0.000000,0.495709,7.050005,0.500666
C_abbayesii_A601_3,56.258536,0.495709,0.000000,7.408919,0.004957
C_abbayesii_A601_4,51.155157,7.050005,7.408919,0.000000,7.412587
C_abbayesii_A601_5,56.263492,0.500666,0.004957,7.412587,0.000000


In [ ]:
# Re-import all necessary packages after kernel reset
import pandas as pd
import numpy as np
from skbio import TreeNode
import itertools

# Reload uploaded files
coords_df = pd.read_csv("../data/dr_guyot_all_for_collection.csv")
genetic_df = pd.read_csv("../data/genetic_dist_matrix.csv", index_col=0)
tree = TreeNode.read("../../input/aligned_caffeine_without_nodes_w_no_coords_tree.nwk")

# Rebuild tree label mapping
tree_labels = genetic_df.index.tolist()
tree_mapping = {}
for label in tree_labels:
    parts = label.split("_")
    if len(parts) >= 3:
        species_name = parts[1].lower()
        tree_mapping[species_name] = label

# Map coordinates to tree labels
coords_df_clean = coords_df.dropna(subset=["latitude", "longitude"]).copy()
coords_df_clean["species_key"] = (
    coords_df_clean["specimen_id"].str.replace("Coffea_", "", regex=False).str.lower()
)
coords_df_clean["tree_label"] = coords_df_clean["species_key"].map(tree_mapping)
matched_coords = coords_df_clean.dropna(subset=["tree_label"])

# Count specimens per tree label
specimen_count = matched_coords["tree_label"].value_counts().to_dict()

# List all tree tips
tip_names = [tip.name for tip in tree.tips()]

# Build new list of specimen-level names
specimen_names = []
for tip_name in tip_names:
    if tip_name in specimen_count:
        for i in range(1, specimen_count[tip_name] + 1):
            specimen_names.append(f"{tip_name}_{i}")

# Compute original tip-level genetic distances
tip_dist_matrix = {}
for tip1, tip2 in itertools.combinations(tree.tips(), 2):
    d = tree.distance(tip1, tip2)
    tip_dist_matrix[(tip1.name, tip2.name)] = d
    tip_dist_matrix[(tip2.name, tip1.name)] = d
for tip in tip_names:
    tip_dist_matrix[(tip, tip)] = 0.0

# Expand to specimen-level distance matrix
specimen_genetic_matrix = pd.DataFrame(
    index=specimen_names, columns=specimen_names, dtype=float
)

for i, s1 in enumerate(specimen_names):
    for j, s2 in enumerate(specimen_names):
        if pd.isna(specimen_genetic_matrix.loc[s1, s2]):
            t1 = "_".join(s1.split("_")[:3])
            t2 = "_".join(s2.split("_")[:3])
            d = tip_dist_matrix.get((t1, t2), 0.0)
            specimen_genetic_matrix.loc[s1, s2] = d
            specimen_genetic_matrix.loc[s2, s1] = d

# Save to file
specimen_genetic_matrix.to_csv("../data/genetic_dist_matrix_full_specimens.csv")
specimen_genetic_matrix.iloc[:5, :5]

""


In [ ]:
# Load the previously generated geographic distance matrix for all specimens
geo_df = pd.read_csv(
    "../data/geographic_dist_matrix_full_specimens_optimized.csv", index_col=0
)

# Reuse these specimen names to replicate the genetic distances accordingly
# We'll use the tree-level distances and duplicate them for each specimen mapping

# Reload tree and extract distances between tip nodes
from Bio import Phylo
from io import StringIO

with open("../../input/aligned_caffeine_without_nodes_w_no_coords_tree.nwk", "r") as f:
    tree_data = f.read()
tree = Phylo.read(StringIO(tree_data), "newick")

# Extract tip distances
from collections import defaultdict
from Bio.Phylo.TreeConstruction import _Matrix


def get_tree_distances(tree):
    terminals = tree.get_terminals()
    distances = defaultdict(dict)
    for i, t1 in enumerate(terminals):
        for j, t2 in enumerate(terminals):
            d = tree.distance(t1, t2)
            distances[t1.name][t2.name] = d
    return distances


tip_distances = get_tree_distances(tree)

# Build a mapping from specimen name to its tree node
specimen_to_tip = {name: "_".join(name.split("_")[:3]) for name in geo_df.index}
unique_specimens = geo_df.index.tolist()

# Build specimen-level genetic distance matrix
genetic_distances = pd.DataFrame(
    index=unique_specimens, columns=unique_specimens, dtype=float
)

for i in unique_specimens:
    for j in unique_specimens:
        tip_i = specimen_to_tip[i]
        tip_j = specimen_to_tip[j]
        genetic_distances.loc[i, j] = tip_distances[tip_i][tip_j]

# Save final specimen-level genetic distance matrix
genetic_distances.to_csv("../data/genetic_dist_matrix_full_specimens.csv")
genetic_distances.iloc[:5, :5]

,C_abbayesii_A601_1,C_abbayesii_A601_2,C_abbayesii_A601_3,C_abbayesii_A601_4,C_abbayesii_A601_5
C_abbayesii_A601_1,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_2,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_3,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_4,0.0,0.0,0.0,0.0,0.0
C_abbayesii_A601_5,0.0,0.0,0.0,0.0,0.0


In [ ]:
import pandas as pd

# Reload necessary files after session reset
coords_df = pd.read_csv("../data/dr_guyot_all_for_collection_renamed.csv")
genetic_df = pd.read_csv("../data/genetic_dist_matrix_full_specimens.csv", index_col=0)

specimen_ids = genetic_df.index.tolist()

# Match directly using the "Renamed_label" column
matched_metadata = coords_df[coords_df["Renamed_label"].isin(specimen_ids)]

# Remove duplicates and keep relevant metadata
final_metadata = matched_metadata[
    ["Renamed_label", "Botanical series", "Habitat class"]
].drop_duplicates()
final_metadata = final_metadata.rename(columns={"Renamed_label": "specimen_id"})

# Save output
output_path = "../data/matched_specimen_metadata_renamed.csv"
final_metadata.to_csv(output_path, index=False)

final_metadata.head()

KeyError: 'Renamed_label'

In [ ]:
import pandas as pd
from skbio.stats.distance import mantel, DistanceMatrix

# Load full specimen-level distance matrices
genetic_df = pd.read_csv("../data/genetic_dist_matrix_full_specimens.csv", index_col=0)
geo_df = pd.read_csv(
    "../data/geographic_dist_matrix_full_specimens_optimized.csv", index_col=0
)

# Load metadata with botanical series information and specimen labels
metadata = pd.read_csv("../data/matched_specimen_metadata_renamed.csv")

# Filter to common specimens
common_specimens = sorted(
    set(genetic_df.index) & set(geo_df.index) & set(metadata["species_name"])
)
genetic_df = genetic_df.loc[common_specimens, common_specimens]
geo_df = geo_df.loc[common_specimens, common_specimens]
metadata = metadata[metadata["specimen_id"].isin(common_specimens)]

# Perform Mantel tests per botanical series
results = []
for series, group in metadata.groupby("Botanical series"):
    specimens = group["specimen_id"].tolist()
    if len(specimens) < 3:
        continue  # skip underpowered groups

    g_dm = DistanceMatrix(genetic_df.loc[specimens, specimens].values, ids=specimens)
    d_dm = DistanceMatrix(geo_df.loc[specimens, specimens].values, ids=specimens)

    r, p, _ = mantel(g_dm, d_dm, method="pearson", permutations=10000)
    results.append(
        {
            "Botanical series": series,
            "n_specimens": len(specimens),
            "Mantel_r": r,
            "p_value": p,
        }
    )

# Convert to DataFrame and display
mantel_df = pd.DataFrame(results).sort_values("p_value")
mantel_df

KeyError: 'specimen_id'